# Profiling Analysis

End-to-end pipeline: extract results → enrich with roofline/BW → visualize scaling.

In [ ]:
# ── Configuration ──

# Experiment run directories (add all runs, later ones override on name collision)
RUN_DIRS = [
    "/data/philei/tt-metal/tt-train/tools/training_estimator/experiments/run_20260224_152552",
    "/data/philei/tt-metal/tt-train/tools/training_estimator/experiments/run_20260224_172829",
]

# Optional enrichment
ADD_ROOFLINE = True
ROOFLINE_DIR = "/data/philei/tt-metal/tt-train-roofline"
ROOFLINE_HARDWARE = "p100"
ROOFLINE_PEAK_TFLOPS = 160.0

ADD_ALLREDUCE_BW = True
AR_TOPOLOGY = "linear"
AR_THEORETICAL_BW = 48.0  # GB/s
AR_TP_MEM_SHARD = 0.88

# Extract settings
SEQ_LEN = 2048
DEVICE_CLOCK_GHZ = 1.35

# Optionally dump final JSON to disk
DUMP_JSON = False
DUMP_PATH = "all_results.json"

In [ ]:
import sys, json
from pathlib import Path

TT_TRAIN = Path("/data/philei/tt-metal/tt-train")
if str(TT_TRAIN / "tools") not in sys.path:
    sys.path.insert(0, str(TT_TRAIN / "tools"))

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

## 1. Extract results to JSON

In [ ]:
import profiling.results_json.extract_results as _ext
_ext.DEVICE_CLOCK_GHZ = DEVICE_CLOCK_GHZ

run_dirs = [Path(d) for d in RUN_DIRS]
output_path = Path(DUMP_PATH) if DUMP_JSON else Path("/tmp/_profiling_results.json")

results = _ext.main_run_dir(run_dirs, output=output_path, seq_len=SEQ_LEN)
print(f"\n{len(results)} experiments loaded")

## 2. (Optional) Enrich with roofline and all-reduce bandwidth

In [ ]:
if ADD_ROOFLINE:
    from profiling.results_json.add_roofline import enrich_roofline
    n = enrich_roofline(results, ROOFLINE_DIR, ROOFLINE_HARDWARE, ROOFLINE_PEAK_TFLOPS)
    print(f"Roofline: {n}/{len(results)} annotated")

if ADD_ALLREDUCE_BW:
    from profiling.results_json.add_allreduce_bw import enrich_allreduce_bw
    n = enrich_allreduce_bw(results, AR_TOPOLOGY, AR_THEORETICAL_BW, AR_TP_MEM_SHARD)
    print(f"All-reduce BW: {n}/{len(results)} annotated")

if DUMP_JSON:
    with open(DUMP_PATH, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved to {DUMP_PATH}")

## 3. Convert to DataFrame

In [ ]:
from profiling.results_visualization.results_to_csv import to_dataframe

df = to_dataframe(results)
print(f"{len(df)} rows, {len(df.columns)} columns")
df

## 4. Visualization

### Batch scaling

In [ ]:
from profiling.results_visualization.plot_batch_scaling import plot_all as plot_batch_all

batch_df = plot_batch_all(df, show=True)
if not batch_df.empty:
    display(batch_df)

### DDP scaling

In [ ]:
from profiling.results_visualization.plot_ddp_scaling import plot_all as plot_ddp_all

ddp_df = plot_ddp_all(df, show=True)
if not ddp_df.empty:
    display(ddp_df)

### TP scaling

In [ ]:
from profiling.results_visualization.plot_tp_scaling import plot_all as plot_tp_all

tp_df = plot_tp_all(df, show=True)
if not tp_df.empty:
    display(tp_df)